# 04 OPD / GKD

Purpose: run teacher-guided OPD/GKD from the configured student checkpoint using a frozen Qwen2.5-7B-Instruct teacher. This public run initializes the student from the selected SFT checkpoint.

Expected inputs: selected SFT checkpoint from `shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected`, `Qwen/Qwen2.5-7B-Instruct`, and FinChain validation/test splits.

Expected outputs: GKD candidate checkpoints, validation-selected model, and eval summaries for `shannan-liu1/qwen25-1p5b-finchain-v2-gkd-selected`.

HF status: selected GKD weights and eval artifacts are in `shannan-liu1/qwen25-1p5b-finchain-v2-gkd-selected`; see `docs/hf_checkpoints.md` for the full list.

## Before you run a single cell in this notebook - terminal pre-flight

This notebook runs on-policy distillation / generalized knowledge distillation (OPD/GKD) on FinChain. Run this terminal block first on a fresh GPU environment. It pins the CUDA/PyTorch stack before NCCL work, keeps HF cache on the persistent volume, requires the FinChain v2 template-disjoint JSONLs and manifest under the local data path, and logs into W&B/Hugging Face before training starts. If you are restarting a stopped environment or recovering from a package failure, rerun this block before resuming training; do not jump straight to an Accelerate launch.

```bash
cd /workspace
test -d finpost || git clone https://github.com/shannan-liu1/finpost.git
cd /workspace/finpost
git checkout main
git pull --ff-only

# Keep package caches and temp files on the persistent volume, not the small container disk.
export HF_HOME=/workspace/hf-cache
export PIP_CACHE_DIR=/workspace/pip-cache
export TMPDIR=/workspace/tmp
export WANDB_DIR=/workspace/wandb
mkdir -p "$HF_HOME" "$PIP_CACHE_DIR" "$TMPDIR" "$WANDB_DIR"

# Install project deps, then fail fast on CUDA/NCCL drift. If the guard fails,
# the repair script removes CUDA 13 pip packages and reinstalls the CUDA 12.4
# Torch CUDA 12.4 stack:
#   torch==2.6.0+cu124
# The repair script also removes optional torchvision/torchaudio wheels;
# finpost does not use them, and broken optional wheels can make transformers imports fail.
python -m pip install -e ".[dev,rlvr,chaineval]"
nvidia-smi || true
bash scripts/repair_cuda_stack.sh
python -m pip install -e ".[dev,rlvr,chaineval]"
python scripts/check_cuda_stack.py

# Persistent cache + FinChain split paths. The public repo does not
# track these JSONLs; generate or place them under data/finchain_v2_template_disjoint
# before the paid GPU run.
export HF_HOME=/workspace/hf-cache
export PIP_CACHE_DIR=${PIP_CACHE_DIR:-/workspace/pip-cache}
export TMPDIR=${TMPDIR:-/workspace/tmp}
export WANDB_DIR=${WANDB_DIR:-/workspace/wandb}
export WANDB_MODE=${WANDB_MODE:-online}
export WANDB_PROJECT=finpost-finchain-gkd
mkdir -p /workspace/data/finchain_v2_template_disjoint "$HF_HOME" "$PIP_CACHE_DIR" "$TMPDIR" "$WANDB_DIR"
for f in train.jsonl validation.jsonl test.jsonl manifest.json; do
  test -f "data/finchain_v2_template_disjoint/$f" || { echo "Missing data/finchain_v2_template_disjoint/$f; generate or place the FinChain v2 splits before training." >&2; exit 1; }
  cp -n "data/finchain_v2_template_disjoint/$f" /workspace/data/finchain_v2_template_disjoint/
done
export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain_v2_template_disjoint/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain_v2_template_disjoint/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain_v2_template_disjoint/test.jsonl
python scripts/audit_finchain_template_disjoint_manifest.py --data-dir data/finchain_v2_template_disjoint --out artifacts/preflight/finchain_v2_manifest_audit.json
python scripts/gpu_preflight.py --out artifacts/preflight/preflight_report.json --timeout-sec 900

# Auth. `wandb status` should show your username. `huggingface-cli whoami`
# should succeed if you plan to push checkpoints or access private/gated repos.
wandb login
wandb status
huggingface-cli login
huggingface-cli whoami

# Pre-download model snapshots so an Accelerate launch does not look hung while
# it is only fetching weights.
HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py Qwen/Qwen2.5-0.5B Qwen/Qwen2.5-1.5B Qwen/Qwen2.5-7B-Instruct
```

Distributed rule of thumb: record distributed training only for cells that actually use `accelerate launch --num_processes {world_size}` or explicit pair-generation sharding. `NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1` is the safe multi-GPU starting point; remove those only after a small distributed canary passes without the NCCL/CUDA-driver error.


# FinChain OPD/GKD GPU Notebook

Trains a configured Qwen2.5-1.5B student on FinChain via on-policy distillation from Qwen2.5-7B-Instruct, following Agarwal et al. 2023 (arXiv 2306.13649). The public run starts from the selected FinChain SFT checkpoint. Default divergence is **forward KL** (paper's arithmetic-reasoning choice). The training step lives in `finpost.training.gkd_train`; the divergence math lives in `finpost.training.gkd`.



## Step 1 - Sanity-check the environment

In [ ]:
from pathlib import Path
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/workspace/finpost") if Path("/workspace/finpost").exists() else PROJECT_ROOT
os.chdir(PROJECT_ROOT)

RESULTS_DIR = PROJECT_ROOT / "results" / "finchain_gkd"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def progress(title, detail=None):
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {title}")
    if detail:
        print(detail)


def run_cmd(cmd, *, check=False):
    progress("running command", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {cmd}")
    return completed


def append_cost_event(stage, **payload):
    path = RESULTS_DIR / "cost_ledger.jsonl"
    row = {"stage": stage, "time": time.strftime("%Y-%m-%dT%H:%M:%S"), **payload}
    with path.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(row, sort_keys=True) + "\n")
    print(json.dumps(row, indent=2, sort_keys=True))
    return row


def attach_cost_ledger(eval_root):
    cost_path = RESULTS_DIR / "cost_ledger.jsonl"
    if not cost_path.exists():
        print("cost ledger not found yet:", cost_path)
        return None
    target = Path(eval_root) / "cost_ledger.jsonl"
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cost_path, target)
    print("attached cost ledger:", target)
    return target


progress("project root", str(PROJECT_ROOT))
progress("python", sys.version.split()[0])
progress("platform", platform.platform())

## Distributed launch preflight

Run this after the setup cell and before any multi-GPU command. If it reports fewer than 2 CUDA devices, stay on the single-GPU path. See `notebooks/01_sft_ablation.ipynb` for the full environment checklist.

In [ ]:
progress("distributed launch preflight")
try:
    import accelerate
    print("accelerate:", accelerate.__version__)
except Exception as exc:
    print("accelerate import failed:", repr(exc))

try:
    import torch
    print("cuda devices:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch cuda check failed:", repr(exc))

for key in ["CUDA_VISIBLE_DEVICES", "WORLD_SIZE", "LOCAL_RANK", "RANK"]:
    print(f"{key}={os.environ.get(key)}")

run_cmd("accelerate env")


## GPU setup guardrails

See `notebooks/01_sft_ablation.ipynb` for the full checklist. OPD/GKD-specific note: `accelerate launch -m finpost.training.gkd_train` wraps the student and dataloader; the frozen teacher is replicated per rank (data parallel, not model sharding).

In [ ]:
import os
import shutil
from pathlib import Path

os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
os.environ.setdefault("PIP_CACHE_DIR", "/workspace/pip-cache")
os.environ.setdefault("TMPDIR", "/workspace/tmp")
os.environ.setdefault("WANDB_DIR", "/workspace/wandb")
os.environ.setdefault("WANDB_MODE", "online")
os.environ.setdefault("WANDB_PROJECT", "finpost-finchain-gkd")

Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["PIP_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["WANDB_DIR"]).mkdir(parents=True, exist_ok=True)

runtime_data_dir = Path("/workspace/data/finchain_v2_template_disjoint")
repo_data_dir = PROJECT_ROOT / "data" / "finchain_v2_template_disjoint"
runtime_data_dir.mkdir(parents=True, exist_ok=True)

missing_inputs = []
for split in ["train", "validation", "test"]:
    source = repo_data_dir / f"{split}.jsonl"
    target = runtime_data_dir / f"{split}.jsonl"
    if not target.exists() and source.exists():
        shutil.copy2(source, target)
    os.environ[f"FINPOST_FINCHAIN_{split.upper()}_JSONL"] = str(target)
    if not target.exists():
        missing_inputs.append(str(target))

manifest_source = repo_data_dir / "manifest.json"
manifest_target = runtime_data_dir / "manifest.json"
if not manifest_target.exists() and manifest_source.exists():
    shutil.copy2(manifest_source, manifest_target)
if not manifest_target.exists():
    missing_inputs.append(str(manifest_target))

if missing_inputs:
    raise FileNotFoundError(
        "Missing FinChain v2 split JSONLs or manifest. Generate or place them under "
        "data/finchain_v2_template_disjoint/ before training: "
        + ", ".join(missing_inputs)
    )

progress("GPU CUDA/data guard")
run_cmd("python scripts/check_cuda_stack.py", check=True)

progress("GKD dependency guard")
run_cmd("python -m finpost.training.gkd_train --help", check=True)
run_cmd("""
python - <<'PY'
from finpost.training.gkd_train import GKDConfig

GKDConfig.model_validate({
    "model": {
        "student_checkpoint": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected",
        "teacher_checkpoint": "Qwen/Qwen2.5-7B-Instruct",
    },
    "training": {"max_steps": 2, "warmup_steps": 0, "lr": 5e-6},
    "checkpointing": {"save_dir": "artifacts/preflight/gkd_config_smoke"},
})
print("GKDConfig OK")
PY
""", check=True)

progress("auth status")
run_cmd("wandb status", check=False)
run_cmd("huggingface-cli whoami", check=False)

for key in [
    "HF_HOME",
    "WANDB_MODE",
    "WANDB_PROJECT",
    "FINPOST_FINCHAIN_TRAIN_JSONL",
    "FINPOST_FINCHAIN_VALIDATION_JSONL",
    "FINPOST_FINCHAIN_TEST_JSONL",
]:
    print(f"{key}={os.environ.get(key)}")


## Step 1.5 - Hugging Face cache warmup

Run once per environment/volume. This downloads tokenizer/config/safetensors into `HF_HOME` without loading the model on GPU. It makes later stalls easier to diagnose: after this cell, a long pause is trainer setup or generation, not model download.


In [ ]:
HF_WARMUP_MODELS = ['Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-1.5B', 'Qwen/Qwen2.5-7B-Instruct']
progress("HF cache warmup", " ".join(HF_WARMUP_MODELS))
warm_cmd = "HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py " + " ".join(HF_WARMUP_MODELS)
run_cmd(warm_cmd, check=True)


In [ ]:
progress("GPU preflight")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch check failed:", repr(exc))

run_cmd("nvidia-smi")

In [ ]:
# Confirm the new GKD modules are importable and FinChain data resolves before GPU work.
for module_name in [
    "finpost.training.gkd",
    "finpost.training.gkd_train",
    "finpost.training.logprobs",
    "finpost.data.finchain_dataset",
    "finpost.evals.finchain_metrics",
]:
    try:
        importlib.import_module(module_name)
        print(module_name, "OK")
    except Exception as exc:
        print(module_name, "FAILED:", repr(exc))

from finpost.data.finchain_dataset import load_finchain, resolve_finchain_path

split_examples = {}
for split in ["train", "validation", "test"]:
    path = resolve_finchain_path(split)
    examples = load_finchain(split)
    split_examples[split] = examples
    print(split, "->", path, "exists=", path.exists(), "rows=", len(examples))

assert len(split_examples["validation"]) == 870, "expected FinChain validation split to contain 870 rows"
assert len(split_examples["test"]) == 870, "expected FinChain test split to contain 870 rows"
train_examples = split_examples["train"]
print("train examples:", len(train_examples))
print("validation examples:", len(split_examples["validation"]))
print("test examples:", len(split_examples["test"]))
print("first prompt id:", train_examples[0].id)
print("first gold answer:", train_examples[0].final_answer)


## Step 2 - Hyperparameters

Primary hyperparameters. The shipped YAMLs at `configs/finchain/gkd/finchain_qwen25_1_5b_canary.yaml` and `configs/finchain/gkd/finchain_qwen25_1_5b.yaml` use the same values; this inline cell keeps the notebook self-contained without requiring config-file edits.

In [ ]:
GKD_CONFIG = {
    # Models
    "student_checkpoint": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected",
    "fast_canary_student_checkpoint": "Qwen/Qwen2.5-0.5B",
    "fast_canary_teacher_checkpoint": "Qwen/Qwen2.5-0.5B",
    "teacher_checkpoint": "Qwen/Qwen2.5-7B-Instruct",
    "student_dtype": "bfloat16",
    "teacher_dtype": "bfloat16",
    "gradient_checkpointing": True,
    "use_8bit_optimizer": False,  # flip to True if a smaller single-GPU environment is memory tight
    # Loss
    "beta": 0.0,  # 0.0 = forward KL (paper default for arithmetic reasoning)
    "sequence_chunk_size": 64,  # exact JSD, lower peak log-softmax VRAM
    # Training
    "max_steps": 500,
    "warmup_steps": 50,
    "lr": 3.0e-6,
    "grad_accum_steps": 2,
    "grad_clip": 1.0,
    "per_device_prompt_batch_size": 1,
    "rollouts_per_prompt": 4,         # K in the paper
    "max_completion_length": 512,     # FinChain rollout cap (generated tokens, not SFT total length)
    "student_temperature": 1.0,       # gamma in the paper
    "checkpoint_every_n_steps": 250,
    "gkd_eval_steps": [250, 500],
    "target_train_prompts_per_step": 4,
    "max_train_gpus": 4,
    "max_eval_gpus": 4,
    "eval_batch_size_finchain": 64,
    # Data
    "max_prompt_len": 512,
    "require_full_prompt_coverage": True,
    "seed": 42,
    # Logging / IO
    "wandb_project": "finpost-finchain-gkd",
    "run_name": "qwen25-1p5b-finchain-v2-gkd",
    "save_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-gkd",
    "hf_selected_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-gkd-selected",
    "hf_candidates_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-gkd-candidates",
    "push_candidates_repo": False,
    "reset_outputs_before_rerun": True,
}
print(json.dumps(GKD_CONFIG, indent=2))


## Step 2.5 - Restore configured OPD/GKD student checkpoint

Run this on a fresh environment before the canary or full run. It restores the configured student checkpoint from Hugging Face into the local path used by this notebook, and skips the download when weights already exist. The public run uses the validation-selected SFT checkpoint.


In [ ]:
from huggingface_hub import snapshot_download

SFT_REPO_ID = "shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected"
student_local_dir = Path(GKD_CONFIG["student_checkpoint"])
SFT_ALLOW_PATTERNS = [
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
    "chat_template.jinja",
    "*.safetensors",
    "*.safetensors.index.json",
    "pytorch_model*.bin",
    "pytorch_model.bin.index.json",
    "vocab.*",
    "merges.txt",
    "*.model",
]

model_files = sorted(student_local_dir.glob("*.safetensors")) + sorted(student_local_dir.glob("pytorch_model*.bin"))
if model_files:
    print("OPD/GKD student checkpoint already present:", student_local_dir)
else:
    print("Restoring OPD/GKD student checkpoint from HF:", SFT_REPO_ID, "->", student_local_dir)
    student_local_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=SFT_REPO_ID,
        repo_type="model",
        local_dir=str(student_local_dir),
        allow_patterns=SFT_ALLOW_PATTERNS,
    )
    model_files = sorted(student_local_dir.glob("*.safetensors")) + sorted(student_local_dir.glob("pytorch_model*.bin"))
    if not model_files:
        raise RuntimeError(f"Downloaded OPD/GKD student checkpoint has no model weights: {student_local_dir}")

print("student model files:", [path.name for path in model_files])


## Step 2.6 - Teacher baseline probe

Run this before OPD/GKD training. OPD only makes sense if the teacher can solve held-out FinChain under the same exact-answer parser. This validation-subset probe compares the configured student against the configured teacher. Raise `TEACHER_PROBE_N` to `870` for the full validation split after the subset probe passes.

In [ ]:
import json
import torch as _t

TEACHER_PROBE_N = 128
TEACHER_PROBE_SPLIT = "validation"
TEACHER_PROBE_ROOT = PROJECT_ROOT / "results" / "evals" / "gkd_teacher_probe_v2_template_disjoint"

teacher_probe_checkpoints = {
    "sft_baseline": GKD_CONFIG["student_checkpoint"],
    "teacher": GKD_CONFIG["teacher_checkpoint"],
    # Optional comparisons. Uncomment deliberately; each 7B candidate costs another eval/download.
    # "qwen25_7b_base": "Qwen/Qwen2.5-7B",
}

teacher_probe_gpu_count = _t.cuda.device_count() if _t.cuda.is_available() else 0
teacher_probe_gpus = [str(i) for i in range(max(1, min(teacher_probe_gpu_count, GKD_CONFIG["max_eval_gpus"])))]
teacher_probe_checkpoint_args = " ".join(
    f"{name}={path}" for name, path in teacher_probe_checkpoints.items()
)
teacher_probe_cmd = (
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {' '.join(teacher_probe_gpus)} "
    "--parallelism examples "
    f"--checkpoints {teacher_probe_checkpoint_args} "
    f"--n {TEACHER_PROBE_N} "
    f"--finchain-split {TEACHER_PROBE_SPLIT} "
    f"--out-dir {TEACHER_PROBE_ROOT} "
    f"--batch-size-finchain {GKD_CONFIG['eval_batch_size_finchain']}"
)
print("teacher probe command:")
print(teacher_probe_cmd)
teacher_probe_result = run_cmd(teacher_probe_cmd, check=True)

teacher_probe_rows = []
for label in teacher_probe_checkpoints:
    summary_path = TEACHER_PROBE_ROOT / label / "accuracy_summary.json"
    rows = json.loads(summary_path.read_text(encoding="utf-8"))
    teacher_probe_rows.extend(rows)
print(json.dumps(teacher_probe_rows, indent=2))

by_label = {row["checkpoint"]: row for row in teacher_probe_rows}
sft_row = by_label.get("sft_baseline")
teacher_row = by_label.get("teacher")
if sft_row and teacher_row:
    delta = float(teacher_row["accuracy"]) - float(sft_row["accuracy"])
    print(f"teacher minus SFT accuracy delta on n={TEACHER_PROBE_N}: {delta:+.6f}")
    if float(teacher_row["parse_success_rate"]) < 0.90:
        print("WARNING: teacher parse success is low; inspect details before OPD/GKD.")
    if delta <= 0.0:
        print("WARNING: configured teacher did not beat the SFT baseline on this probe.")


## Step 3 - Build YAML configs

Writes generated runtime YAMLs under `artifacts/generated_configs/`: a canary config (20 steps, batch 1, K=2, completion length 96 - fast smoke) and a full config (the values above).

In [ ]:
import yaml

GENERATED_CONFIG_DIR = PROJECT_ROOT / "artifacts" / "generated_configs" / "gkd"
GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)


def render_config(cfg, *, max_steps, max_completion_length, rollouts, ckpt_every, save_dir, run_name):
    return {
        "model": {
            "student_checkpoint": cfg["student_checkpoint"],
            "teacher_checkpoint": cfg["teacher_checkpoint"],
            "student_dtype": cfg["student_dtype"],
            "teacher_dtype": cfg["teacher_dtype"],
            "use_safetensors": True,
            "gradient_checkpointing": cfg["gradient_checkpointing"],
            "use_8bit_optimizer": cfg["use_8bit_optimizer"],
        },
        "data": {
            "source": "finchain",
            "split": "train",
            "max_prompt_len": cfg["max_prompt_len"],
            "require_full_prompt_coverage": cfg["require_full_prompt_coverage"],
            "seed": cfg["seed"],
        },
        "gkd": {"beta": cfg["beta"], "sequence_chunk_size": cfg["sequence_chunk_size"]},
        "training": {
            "max_steps": max_steps,
            "warmup_steps": min(cfg["warmup_steps"], max_steps - 1),
            "lr": cfg["lr"],
            "weight_decay": 0.01,
            "grad_accum_steps": cfg["grad_accum_steps"],
            "grad_clip": cfg["grad_clip"],
            "checkpoint_every_n_steps": ckpt_every,
            "per_device_prompt_batch_size": cfg["per_device_prompt_batch_size"],
            "rollouts_per_prompt": rollouts,
            "max_completion_length": max_completion_length,
            "student_temperature": cfg["student_temperature"],
            "dataloader_num_workers": 0,
            "pin_memory": True,
        },
        "logging": {
            "wandb_project": cfg["wandb_project"],
            "run_name": run_name,
        },
        "checkpointing": {
            "save_dir": save_dir,
            "retention_last_n": 3,
            "resume_from": None,
        },
    }


canary_yaml = GENERATED_CONFIG_DIR / "finchain_qwen25_0_5b_canary.generated.yaml"
full_yaml = GENERATED_CONFIG_DIR / "finchain_qwen25_1_5b.generated.yaml"

fast_canary_config = dict(GKD_CONFIG)
fast_canary_config["student_checkpoint"] = GKD_CONFIG["fast_canary_student_checkpoint"]
fast_canary_config["teacher_checkpoint"] = GKD_CONFIG["fast_canary_teacher_checkpoint"]

canary_yaml.write_text(
    yaml.safe_dump(
        render_config(
            fast_canary_config,
            max_steps=20,
            max_completion_length=96,
            rollouts=2,
            ckpt_every=20,
            save_dir=GKD_CONFIG["save_dir"] + "-canary-0p5b",
            run_name=GKD_CONFIG["run_name"] + "-canary-0p5b",
        ),
        sort_keys=False,
    ),
    encoding="utf-8",
)
full_yaml.write_text(
    yaml.safe_dump(
        render_config(
            GKD_CONFIG,
            max_steps=GKD_CONFIG["max_steps"],
            max_completion_length=GKD_CONFIG["max_completion_length"],
            rollouts=GKD_CONFIG["rollouts_per_prompt"],
            ckpt_every=GKD_CONFIG["checkpoint_every_n_steps"],
            save_dir=GKD_CONFIG["save_dir"],
            run_name=GKD_CONFIG["run_name"],
        ),
        sort_keys=False,
    ),
    encoding="utf-8",
)
print("wrote", canary_yaml)
print("wrote", full_yaml)


## Step 3.5 - Canary: 20-step smoke test before the full run

Catches:
- Teacher OOM on this hardware (Qwen2.5-7B in bf16 is ~15 GB on top of the student).
- NaN loss from divergence math (forward KL has +infinity failure modes; the canary surfaces them before the full run).
- Broken generation (wrong pad/eos handling, label mask leak).
- FinChain prompt loader misconfigured (the most common silent failure).

Should finish in ~10 minutes. If anything fails here, **do not run the full cell below**. Diagnose first.

In [ ]:
canary_start = time.perf_counter()
canary_result = run_cmd(
    f"python -m finpost.training.gkd_train --config {canary_yaml}",
    check=False,
)
canary_elapsed = time.perf_counter() - canary_start
append_cost_event(
    "gkd_canary_fast_0p5b",
    elapsed_sec=round(canary_elapsed, 1),
    exit_code=canary_result.returncode,
    config=str(canary_yaml),
)
if canary_result.returncode != 0:
    raise RuntimeError(
        f"Canary failed with exit {canary_result.returncode}. "
        "Do NOT continue to the full run until this is fixed."
    )


In [ ]:
# Disk + GPU snapshot after the canary. Use this to plan the full run.
run_cmd("df -h")
run_cmd("nvidia-smi")
run_cmd(f"ls -lh {GKD_CONFIG['save_dir']}-canary-0p5b/")

## Step 4 - Full GKD training run

Wall clock estimate: ~3-4 hours on a single 48 GB GPU at the default config. wandb stream will show loss, mean_jsd, response_tokens_per_sec, grad_norm.

## Step 4.5 - Full OPD/GKD training run

Default path: OPD/GKD via `accelerate launch --num_processes {gkd_world_size}` when multiple CUDA devices are visible, capped by `max_train_gpus`. The frozen Qwen2.5-7B-Instruct teacher is replicated per rank, so this is data-parallel training, not model sharding. Fallback to single-process training only when the environment exposes fewer than two compatible GPUs.


In [ ]:
import shutil
import torch as _t
import yaml

if GKD_CONFIG["reset_outputs_before_rerun"]:
    for reset_path in [
        Path(GKD_CONFIG["save_dir"]),
        Path(f"{GKD_CONFIG['save_dir']}-hf-candidates"),
        Path(f"{GKD_CONFIG['save_dir']}-selected"),
        PROJECT_ROOT / "results" / "evals" / "gkd_v2_template_disjoint",
    ]:
        if reset_path.exists():
            print("removing stale artifact:", reset_path)
            shutil.rmtree(reset_path)

gpu_count = _t.cuda.device_count() if _t.cuda.is_available() else 0
gkd_visible_world_size = max(1, min(gpu_count, GKD_CONFIG["max_train_gpus"]))
gkd_per_device_prompt_batch_size = GKD_CONFIG["per_device_prompt_batch_size"]
gkd_target_train_prompts = GKD_CONFIG.get("target_train_prompts_per_step")
if gkd_visible_world_size > 1 and gkd_target_train_prompts is not None:
    compatible_world_sizes = [
        world_size
        for world_size in range(gkd_visible_world_size, 0, -1)
        if gkd_target_train_prompts % (world_size * gkd_per_device_prompt_batch_size) == 0
    ]
    if not compatible_world_sizes:
        raise RuntimeError(
            "target_train_prompts_per_step must be divisible by at least one "
            f"compatible distributed prompt microbatch. target={gkd_target_train_prompts}, "
            f"visible_world_size={gkd_visible_world_size}, "
            f"per_device_prompt_batch_size={gkd_per_device_prompt_batch_size}"
        )
    gkd_world_size = compatible_world_sizes[0]
    gkd_global_prompt_microbatch = gkd_world_size * gkd_per_device_prompt_batch_size
    gkd_grad_accum_steps = max(1, gkd_target_train_prompts // gkd_global_prompt_microbatch)
else:
    gkd_world_size = gkd_visible_world_size
    gkd_global_prompt_microbatch = gkd_world_size * gkd_per_device_prompt_batch_size
    gkd_grad_accum_steps = GKD_CONFIG["grad_accum_steps"]

gkd_run_config = dict(GKD_CONFIG)
gkd_run_config["grad_accum_steps"] = gkd_grad_accum_steps
full_yaml.write_text(
    yaml.safe_dump(
        render_config(
            gkd_run_config,
            max_steps=GKD_CONFIG["max_steps"],
            max_completion_length=GKD_CONFIG["max_completion_length"],
            rollouts=GKD_CONFIG["rollouts_per_prompt"],
            ckpt_every=GKD_CONFIG["checkpoint_every_n_steps"],
            save_dir=GKD_CONFIG["save_dir"],
            run_name=GKD_CONFIG["run_name"],
        ),
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(
    "OPD/GKD train sizing:",
    {
        "visible_gpus": gpu_count,
        "world_size": gkd_world_size,
        "per_device_prompt_batch_size": gkd_per_device_prompt_batch_size,
        "global_prompt_microbatch": gkd_global_prompt_microbatch,
        "grad_accum_steps": gkd_grad_accum_steps,
        "effective_train_prompts_per_step": gkd_global_prompt_microbatch * gkd_grad_accum_steps,
    },
)
if gkd_world_size != gkd_visible_world_size:
    print(f"Using {gkd_world_size} compatible GPUs out of {gkd_visible_world_size} visible/capped GPUs.")

if gkd_world_size > 1:
    gkd_full_cmd = (
        "HF_HOME=/workspace/hf-cache NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 "
        f"accelerate launch --num_processes {gkd_world_size} --mixed_precision bf16 "
        "-m finpost.training.gkd_train "
        f"--config {full_yaml}"
    )
    cost_label = f"gkd_full_{gkd_world_size}_gpu"
else:
    gkd_full_cmd = f"python -m finpost.training.gkd_train --config {full_yaml}"
    cost_label = "gkd_full_single_process"
    print("WARNING: fewer than 2 CUDA devices visible; falling back to single-process OPD/GKD.")

full_start = time.perf_counter()
full_result = run_cmd(gkd_full_cmd, check=True)
full_elapsed = time.perf_counter() - full_start
append_cost_event(
    cost_label,
    elapsed_sec=round(full_elapsed, 1),
    exit_code=full_result.returncode,
    config=str(full_yaml),
)


## Step 5 - Verify checkpoints landed

In [ ]:
save_dir = Path(GKD_CONFIG["save_dir"])
if not save_dir.exists():
    raise FileNotFoundError(f"save_dir does not exist: {save_dir}")
all_step_dirs = sorted(save_dir.glob("step-*"))
print("all step dirs:", [path.name for path in all_step_dirs])
step_dirs = []
for step in GKD_CONFIG["gkd_eval_steps"]:
    step_dir = save_dir / f"step-{step:08d}"
    has_model = (step_dir / "model.safetensors").exists()
    has_state = (step_dir / "state.pt").exists()
    print(f"{step_dir.name}: model={has_model} state={has_state}")
    if not step_dir.exists() or not has_model or not has_state:
        raise RuntimeError(f"Expected fresh GKD checkpoint is missing or incomplete: {step_dir}")
    step_dirs.append(step_dir)
expected_labels = {
    f"gkd_step_{step:08d}"
    for step in GKD_CONFIG["gkd_eval_steps"]
}
print("expected candidate labels:", sorted(expected_labels))

## Step 6 - Convert candidates, select OPD/GKD on validation, then test once

The OPD/GKD trainer writes repo-native `step-*` checkpoints. Convert every
saved candidate to HF format, select only by validation `accuracy`, and run
the selected student versus the fixed SFT baseline on test with ChainEval.


In [ ]:
import shutil

converted_root = Path(f"{GKD_CONFIG['save_dir']}-hf-candidates")
gkd_candidate_dirs = {}
for raw_dir in step_dirs:
    converted_dir = converted_root / raw_dir.name
    run_cmd(
        "python scripts/convert_checkpoint_to_hf.py "
        f"--checkpoint-dir {raw_dir} --base-model-id {GKD_CONFIG['student_checkpoint']} "
        f"--out-dir {converted_dir} --dtype bfloat16",
        check=True,
    )
    gkd_candidate_dirs[f"gkd_{raw_dir.name.replace('-', '_')}"] = converted_dir

if set(gkd_candidate_dirs) != expected_labels:
    raise RuntimeError(
        f"GKD candidate set mismatch: got {sorted(gkd_candidate_dirs)}, expected {sorted(expected_labels)}"
    )

EVAL_ROOT = PROJECT_ROOT / "results" / "evals" / "gkd_v2_template_disjoint"
VALIDATION_OUT_DIR = EVAL_ROOT / "validation_selection"
TEST_OUT_DIR = EVAL_ROOT / "test_selected_chaineval"
EVAL_CONFIG = {
    "parallelism": "examples",
    "batch_size_finchain": GKD_CONFIG["eval_batch_size_finchain"],
    "max_eval_gpus": GKD_CONFIG["max_eval_gpus"],
    "chaineval_batch_size": 8,
}
try:
    import torch
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    gpu_count = 0
gpus_arg = " ".join(str(idx) for idx in range(max(1, min(gpu_count, EVAL_CONFIG["max_eval_gpus"]))))
candidate_args = " ".join(f"{label}={path}" for label, path in gkd_candidate_dirs.items())
validation_checkpoint_args = f"sft_baseline={GKD_CONFIG['student_checkpoint']} {candidate_args}"

run_cmd(
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints {validation_checkpoint_args} --finchain-split validation --n 870 "
    f"--out-dir {VALIDATION_OUT_DIR} --batch-size-finchain {EVAL_CONFIG['batch_size_finchain']}",
    check=True,
)
validation_summaries = {
    label: json.loads((VALIDATION_OUT_DIR / label / "accuracy_summary.json").read_text(encoding="utf-8"))[0]
    for label in ["sft_baseline", *gkd_candidate_dirs]
}
gkd_validation_summaries = {
    label: validation_summaries[label]
    for label in gkd_candidate_dirs
}
selected_label = sorted(
    gkd_validation_summaries,
    key=lambda label: (
        -gkd_validation_summaries[label]["accuracy"],
        -gkd_validation_summaries[label].get("parse_success_rate", 0.0),
        int(label.rsplit("_", 1)[-1]),
    ),
)[0]
sft_validation_accuracy = validation_summaries["sft_baseline"]["accuracy"]
selected_validation_accuracy = gkd_validation_summaries[selected_label]["accuracy"]
if selected_validation_accuracy <= sft_validation_accuracy:
    raise RuntimeError(
        "OPD/GKD failed validation gate: best GKD checkpoint "
        f"{selected_label} accuracy={selected_validation_accuracy:.6f} did not beat "
        f"sft_baseline accuracy={sft_validation_accuracy:.6f}. Do not run test or push."
    )
gkd_selected_dir = Path(f"{GKD_CONFIG['save_dir']}-selected")
if gkd_selected_dir.exists():
    shutil.rmtree(gkd_selected_dir)
shutil.copytree(gkd_candidate_dirs[selected_label], gkd_selected_dir)
print("selected on validation:", selected_label, gkd_selected_dir)

selection_metadata = {
    "selection_split": "validation",
    "final_eval_split": "test",
    "selected_checkpoint": str(gkd_selected_dir),
    "selected_label": selected_label,
    "selection_metric": "accuracy",
    "sft_validation_accuracy": sft_validation_accuracy,
    "selected_validation_accuracy": selected_validation_accuracy,
    "validation_summaries": validation_summaries,
    "tie_breaker": "parse_success_rate_then_earliest_step",
    "test_touched_before_selection": False,
}
(EVAL_ROOT / "selection_metadata.json").write_text(
    json.dumps(selection_metadata, indent=2, sort_keys=True),
    encoding="utf-8",
)

run_cmd(
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints sft_baseline={GKD_CONFIG['student_checkpoint']} gkd_selected={gkd_selected_dir} "
    f"--finchain-split test --n 870 --out-dir {TEST_OUT_DIR} "
    f"--batch-size-finchain {EVAL_CONFIG['batch_size_finchain']} --enable-chaineval "
    f"--chaineval-batch-size {EVAL_CONFIG['chaineval_batch_size']}",
    check=True,
)
append_cost_event("gkd_v2_selected_test_chaineval", selection=selected_label)


## Step 7 - Headline numbers

Do not choose GKD from test. `validation_summaries` records checkpoint
selection; the test comparison reports final-answer `accuracy` plus ChainEval
chain precision, recall, F1, and alignment diagnostics for the selected model.


In [ ]:
BASELINE_AND_SELECTED_LABELS = ["sft_baseline", "gkd_selected"]
HEADLINE_METRICS = ["n", "accuracy", "parse_success_rate", "step_recall", "step_precision", "step_f1"]

def compact_metrics(summary):
    return {metric: summary.get(metric) for metric in HEADLINE_METRICS if metric in summary}

test_summaries = {}
for name in BASELINE_AND_SELECTED_LABELS:
    summary_path = TEST_OUT_DIR / name / "accuracy_summary.json"
    test_summaries[name] = json.loads(summary_path.read_text(encoding="utf-8"))[0]
print("validation selection:")
print(json.dumps(validation_summaries, indent=2, sort_keys=True))
print("untouched test summary metrics:")
print(json.dumps({name: compact_metrics(summary) for name, summary in test_summaries.items()}, indent=2, sort_keys=True))
print("untouched test full ChainEval summaries:")
print(json.dumps(test_summaries, indent=2, sort_keys=True))


## Push the OPD/GKD checkpoint to HF Hub

This uploads the HF-format model folder to `shannan-liu1/qwen25-1p5b-finchain-v2-gkd-selected`. Run only after the checkpoint smoke-load/eval cell succeeds and `huggingface-cli whoami` shows the account that can write to `shannan-liu1`.


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

api = HfApi()
repo_id = GKD_CONFIG["hf_selected_repo"]
folder_to_push = Path(gkd_selected_dir)
if not folder_to_push.exists():
    raise FileNotFoundError(f"checkpoint folder does not exist: {folder_to_push}")

api.create_repo(repo_id, exist_ok=True, private=False)
existing_files = [
    path
    for path in api.list_repo_files(repo_id, repo_type="model")
    if path != ".gitattributes"
]
for repo_file in existing_files:
    try:
        api.delete_file(
            path_in_repo=repo_file,
            repo_id=repo_id,
            repo_type="model",
            commit_message=f"Remove stale OPD/GKD selected file {repo_file}",
        )
    except HfHubHTTPError as exc:
        print("delete skipped", repo_file, repr(exc))

api.upload_folder(
    folder_path=str(folder_to_push),
    repo_id=repo_id,
    repo_type="model",
    commit_message="FinChain v2 OPD/GKD validation-selected HF checkpoint",
)
api.upload_folder(
    folder_path=str(EVAL_ROOT),
    repo_id=repo_id,
    repo_type="model",
    path_in_repo="evals/gkd_v2_template_disjoint",
    commit_message="Upload OPD/GKD validation/test eval artifacts",
)
print(f"pushed {folder_to_push} to https://huggingface.co/{repo_id}")
append_cost_event("hf_push_opd_gkd_v2_selected", repo_id=repo_id, folder=str(folder_to_push), eval_root=str(EVAL_ROOT))


## Optional: push OPD/GKD candidate checkpoints to HF Hub

Run this only after selected-checkpoint upload succeeds and the candidates repo is verified in `docs/hf_checkpoints.md`. It is default-off because the reported OPD/GKD results use the selected repo and eval artifacts above.


In [ ]:
PUSH_GKD_CANDIDATES = GKD_CONFIG.get("push_candidates_repo", False)
if not PUSH_GKD_CANDIDATES:
    print("Skipping OPD/GKD candidate checkpoint upload. Set GKD_CONFIG[\"push_candidates_repo\"]=True after verifying the candidates repo in docs/hf_checkpoints.md.")
else:
    from huggingface_hub import HfApi
    from huggingface_hub.utils import HfHubHTTPError
    
    api = HfApi()
    candidates_repo_id = GKD_CONFIG["hf_candidates_repo"]
    api.create_repo(candidates_repo_id, exist_ok=True, private=False)
    attach_cost_ledger(EVAL_ROOT)
    
    for folder in [
        "step-00000250",
        "step-00000500",
        "step-00000750",
        "step-00001000",
        "step-00001250",
        "final",
        "selected",
        "evals",
    ]:
        try:
            api.delete_folder(
                path_in_repo=folder,
                repo_id=candidates_repo_id,
                repo_type="model",
                commit_message=f"Remove stale OPD/GKD candidate path {folder}",
            )
            print("deleted stale HF path:", folder)
        except HfHubHTTPError as exc:
            print("delete skipped", folder, repr(exc))
    
    for label, path in gkd_candidate_dirs.items():
        api.upload_folder(
            folder_path=str(path),
            repo_id=candidates_repo_id,
            repo_type="model",
            path_in_repo=path.name,
            commit_message=f"Upload OPD/GKD candidate {path.name}",
        )
    
    api.upload_folder(
        folder_path=str(EVAL_ROOT),
        repo_id=candidates_repo_id,
        repo_type="model",
        path_in_repo="evals/gkd_v2_template_disjoint",
        commit_message="Upload OPD/GKD validation/test eval and cost artifacts",
    )
    
    print(f"uploaded OPD/GKD candidates/evals to https://huggingface.co/{candidates_repo_id}")
    append_cost_event(
        "hf_push_opd_gkd_v2_candidates_evals",
        repo_id=candidates_repo_id,
        candidates=list(gkd_candidate_dirs),
        eval_root=str(EVAL_ROOT),
    )
    cost_ledger_path = attach_cost_ledger(EVAL_ROOT)
    if cost_ledger_path is not None:
        api.upload_file(
            path_or_fileobj=str(cost_ledger_path),
            repo_id=candidates_repo_id,
            repo_type="model",
            path_in_repo="evals/gkd_v2_template_disjoint/cost_ledger.jsonl",
            commit_message="Update OPD/GKD cost ledger after HF candidate upload",
        )


## Final - Stop the instance

GPU rentals can charge by the hour. If this notebook reached this point and you don't need the GPU anymore, stop the instance through your provider console or CLI. The `results/` directory and any HF push you made above persist outside the instance.